In [0]:
from pyspark.sql import functions as F

CATALOG_SCHEMA = "workspace.stocks"
RAW            = "/Volumes/workspace/stocks/raw"
PRICES_PATH    = f"{RAW}/prices/"
COMPANIES_PATH = f"{RAW}/reference/companies.csv"
CHECKPOINT     = f"{RAW}/_checkpoints/bronze_prices"
SCHEMA_LOC     = f"{RAW}/_schemas/prices"

In [0]:
q = (spark.readStream.format("cloudFiles")
  .option("cloudFiles.format", "json")
  .option("cloudFiles.schemaLocation", SCHEMA_LOC)
  .load(PRICES_PATH)
  .withColumn("load_ts", F.current_timestamp())
  .withColumn("source_file", F.col("_metadata.file_path"))
  .writeStream
  .option("checkpointLocation", CHECKPOINT)
  .trigger(availableNow=True)
  .toTable(f"{CATALOG_SCHEMA}.bronze_prices"))


q.awaitTermination()

In [0]:
(spark.read
  .option("header", True)
  .csv(COMPANIES_PATH)
  .withColumn("load_ts", F.current_timestamp())
  .withColumn("source_file", F.col("_metadata.file_path"))
  .write.format("delta")
  .mode("overwrite")
  .saveAsTable(f"{CATALOG_SCHEMA}.bronze_companies"))

In [0]:
bp = spark.table(f"{CATALOG_SCHEMA}.bronze_prices")
print("bronze_prices rows:", bp.count())
print("tickers:", bp.select("ticker").distinct().count())
print("trade dates:", bp.select("trade_date").distinct().count())
print("rescued rows:", bp.filter(F.col("_rescued_data").isNotNull()).count())
bp.select("ticker", "trade_date", "close", "load_ts", "source_file").show(3, truncate=False)
print("bronze_companies rows:", spark.table(f"{CATALOG_SCHEMA}.bronze_companies").count())

bronze_prices rows: 3201
tickers: 8
trade dates: 105
rescued rows: 0
+------+----------+--------+-----------------------+-----------------------------------------------------------------+
|ticker|trade_date|close   |load_ts                |source_file                                                      |
+------+----------+--------+-----------------------+-----------------------------------------------------------------+
|NVDA  |2026-10-27|217.9400|2026-10-29 06:25:22.749|/Volumes/workspace/stocks/raw/prices/prices_NVDA_2026-10-28.jsonl|
|NVDA  |2026-10-26|216.0000|2026-10-29 06:25:22.749|/Volumes/workspace/stocks/raw/prices/prices_NVDA_2026-10-28.jsonl|
|NVDA  |2026-10-23|210.9600|2026-10-29 06:25:22.749|/Volumes/workspace/stocks/raw/prices/prices_NVDA_2026-10-28.jsonl|
+------+----------+--------+-----------------------+-----------------------------------------------------------------+
only showing top 3 rows
bronze_companies rows: 8
